# Day 3B · Agent Memory（Colab + NVIDIA API 版）

这个 notebook 是把原始 `day-3b-agent-memory.ipynb` 改成 **Google Colab + NVIDIA API** 可直接运行的版本。

本版本做了这些改动：

- 把 Kaggle Secret 改成了 **Colab Secret / 手动输入**
- 把 `Gemini(...)` 改成了 **ADK + LiteLlm + NVIDIA OpenAI-compatible endpoint**
- 记忆检索工具改成了当前官方文档形式：`LoadMemoryTool()` / `PreloadMemoryTool()`

运行建议：

1. 从上到下顺序运行  
2. 先完成手动 `add_session_to_memory()` 的部分  
3. 再运行 callback + `PreloadMemoryTool()` 的自动记忆演示

In [1]:
# Step 0: 安装依赖
!python -m pip install -q -U google-adk litellm openai requests pandas python-dotenv

print("✅ Dependencies installed.")

✅ Dependencies installed.


In [2]:
# Step 1: 读取 NVIDIA API Key
import os
import getpass

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
    if not NVIDIA_API_KEY:
        raise ValueError("Colab Secret 'NVIDIA_API_KEY' is empty.")
    print("✅ Loaded NVIDIA_API_KEY from Colab Secrets.")
except Exception as e:
    print(f"⚠️ Could not read Colab Secret automatically: {e}")
    NVIDIA_API_KEY = getpass.getpass("Please paste your NVIDIA_API_KEY here: ").strip()
    print("✅ Loaded NVIDIA_API_KEY from manual input.")

NVIDIA_API_BASE = "https://integrate.api.nvidia.com/v1"

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
os.environ["OPENAI_API_KEY"] = NVIDIA_API_KEY

print("✅ Environment variables prepared.")
print("NVIDIA_API_BASE =", NVIDIA_API_BASE)

⚠️ Could not read Colab Secret automatically: No module named 'google.colab'
✅ Loaded NVIDIA_API_KEY from manual input.
✅ Environment variables prepared.
NVIDIA_API_BASE = https://integrate.api.nvidia.com/v1


In [3]:
# Step 2: 列出当前 NVIDIA endpoint 可用的模型
import requests
import pandas as pd

headers = {"Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}"}
resp = requests.get(f"{NVIDIA_API_BASE}/models", headers=headers, timeout=30)
resp.raise_for_status()

models_payload = resp.json()
model_rows = models_payload.get("data", [])
model_ids = [row.get("id") for row in model_rows if row.get("id")]

print(f"✅ Found {len(model_ids)} models from NVIDIA endpoint.")
print("First 20 model ids:")
for mid in model_ids[:20]:
    print(" -", mid)

df_models = pd.DataFrame(model_rows)
df_models.head()

✅ Found 136 models from NVIDIA endpoint.
First 20 model ids:
 - 01-ai/yi-large
 - abacusai/dracarys-llama-3.1-70b-instruct
 - adept/fuyu-8b
 - ai21labs/jamba-1.5-large-instruct
 - aisingapore/sea-lion-7b-instruct
 - baai/bge-m3
 - bigcode/starcoder2-15b
 - bytedance/seed-oss-36b-instruct
 - databricks/dbrx-instruct
 - deepseek-ai/deepseek-coder-6.7b-instruct
 - deepseek-ai/deepseek-v3.1-terminus
 - deepseek-ai/deepseek-v3.2
 - deepseek-ai/deepseek-v4-flash
 - deepseek-ai/deepseek-v4-pro
 - google/codegemma-1.1-7b
 - google/codegemma-7b
 - google/deplot
 - google/gemma-2-2b-it
 - google/gemma-2b
 - google/gemma-3-12b-it


,id,object,created,owned_by
0,01-ai/yi-large,model,735790403,01-ai
1,abacusai/dracarys-llama-3.1-70b-instruct,model,735790403,abacusai
2,adept/fuyu-8b,model,735790403,adept
3,ai21labs/jamba-1.5-large-instruct,model,735790403,ai21labs
4,aisingapore/sea-lion-7b-instruct,model,735790403,aisingapore


In [4]:
# Step 3: 自动选择一个更适合 tool / agent 场景的模型
# 按优先级尝试，选择你账户有权限访问的第一个
preferred_candidates = [
    "openai/gpt-oss-120b",                    # 优先：开源 GPT 级别
    "nvidia/llama-3.3-nemotron-super-49b-v1.5",  # Nemotron 49B
    "meta/llama-3.1-70b-instruct",            # Llama 3.1 70B
    "meta/llama-3.1-8b-instruct",             # Llama 3.1 8B (兜底)
]

NVIDIA_MODEL_NAME = None
for candidate in preferred_candidates:
    if candidate in model_ids:
        NVIDIA_MODEL_NAME = candidate
        break

if NVIDIA_MODEL_NAME is None:
    if not model_ids:
        raise RuntimeError("No models returned by NVIDIA /v1/models.")
    # 兜底：用列表里第一个可用模型
    NVIDIA_MODEL_NAME = model_ids[0]

MODEL_NAME = NVIDIA_MODEL_NAME
MODEL_LITELLM = f"openai/{NVIDIA_MODEL_NAME}"

print("✅ Selected NVIDIA model:", NVIDIA_MODEL_NAME)
print("✅ LiteLLM model string:", MODEL_LITELLM)


✅ Selected NVIDIA model: openai/gpt-oss-120b
✅ LiteLLM model string: openai/openai/gpt-oss-120b


In [5]:
# 🔍 诊断：确认 API Key 来源和权限
import os

key = os.environ.get("NVIDIA_API_KEY", "")
print(f"Key 长度: {len(key)}")
print(f"Key 前缀: {key[:20]}...")

# 测试1: 查模型列表 (之前成功的)
import requests
headers = {"Authorization": f"Bearer {key}"}
resp_list = requests.get("https://integrate.api.nvidia.com/v1/models", headers=headers)
print(f"1️⃣ GET /models: {resp_list.status_code}")

# 测试2: 直接调用 gpt-oss-120b (会失败的)
resp_chat = requests.post(
    "https://integrate.api.nvidia.com/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "openai/gpt-oss-120b",
        "messages": [{"role": "user", "content": "Hello"}],
        "max_tokens": 10,
    }
)
print(f"2️⃣ POST /chat/completions: {resp_chat.status_code}")
print(f"   错误详情: {resp_chat.text[:200]}")


Key 长度: 70
Key 前缀: nvapi-OJ_0YE6kWvX4m6...
1️⃣ GET /models: 200
2️⃣ POST /chat/completions: 200
   错误详情: {"id":"chatcmpl-946762e981098a44","object":"chat.completion","created":1777347153,"model":"openai/gpt-oss-120b","choices":[{"index":0,"message":{"role":"assistant","content":null,"refusal":null,"annot


In [6]:
# Step 4: 先直连测试 NVIDIA API
from openai import OpenAI

client = OpenAI(
    base_url=NVIDIA_API_BASE,
    api_key=os.environ["NVIDIA_API_KEY"],
)

test_response = client.chat.completions.create(
    model=NVIDIA_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Reply with one sentence: confirm the API connection is working."},
    ],
    temperature=0.2,
    max_tokens=80,
)

print("✅ Direct NVIDIA API test succeeded.")
print(test_response.choices[0].message.content)

✅ Direct NVIDIA API test succeeded.
The API connection is working correctly.


In [7]:
# Step 5: 导入 ADK 组件
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools.load_memory_tool import LoadMemoryTool
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.genai import types

print("✅ ADK components imported successfully.")

/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


✅ ADK components imported successfully.


/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [8]:
# ============================================
# Step 6: 公共配置与辅助函数
# ============================================
# 本模块定义了 notebook 全局共享的配置常量、模型工厂函数和会话运行辅助函数。
# 这些是整个 Agent Memory 演示的基础设施，被后续所有 cell 复用。

# ─────────────────────────────────────────────
# 1. HTTP 重试配置（当前定义但未绑定，预留扩展）
# ─────────────────────────────────────────────
# HttpRetryOptions 是 google.genai.types 提供的重试策略配置类。
# 当 LLM API 返回特定状态码时，ADK 底层会自动按指数退避策略重试请求。
retry_config = types.HttpRetryOptions(
    attempts=5,              # 最多重试 5 次
    exp_base=7,              # 指数退避的底数：delay = 7^n，增长非常快
                             # 第1次重试等7秒，第2次49秒，第3次343秒...
    initial_delay=1,         # 首次重试前的初始等待时间（秒）
    http_status_codes=[      # 触发重试的 HTTP 状态码白名单
        429,                 # Too Many Requests — 速率限制，必重试
        500,                 # Internal Server Error — 服务器临时故障
        503,                 # Service Unavailable — 服务不可用
        504,                 # Gateway Timeout — 网关超时
    ],
)
# ⚠️ 注意：当前 retry_config 只是定义了变量，还没有实际绑定到 model 或 runner 上。
# 如果要启用，需要在 LiteLlm() 或 Runner() 的初始化参数中传入。

# ─────────────────────────────────────────────
# 2. 全局常量配置
# ─────────────────────────────────────────────
# ADK 使用 (app_name, user_id, session_id) 三元组来唯一标识一个会话。
# 这里把 app_name 和 user_id 固定为常量，后续只通过 session_id 区分不同对话。

APP_NAME = "MemoryDemoApp"   # 应用名称：同一个 app 下的 session 共享配置空间
USER_ID = "demo_user"        # 用户标识：多用户场景下用来隔离不同用户的数据

# ─────────────────────────────────────────────
# 3. 模型工厂函数
# ─────────────────────────────────────────────
# 工厂模式：封装 LiteLlm 的创建逻辑，避免在每个 Agent 定义处重复写相同参数。
# 返回类型 -> LiteLlm 明确告诉调用者和 IDE：这个函数产出的是 ADK 兼容的模型包装器。
def make_nvidia_model() -> LiteLlm:
    """
    创建并返回一个配置好的 LiteLlm 实例，用于连接 NVIDIA OpenAI-compatible API。
    
    LiteLlm 是 ADK 对 LiteLLM 库的封装，让 ADK 能调用任意 OpenAI-compatible endpoint。
    这里通过环境变量动态注入 API Key，避免硬编码敏感信息。
    """
    return LiteLlm(
        model=MODEL_LITELLM,              # 模型标识字符串，如 "openai/openai/gpt-oss-120b"
                                          # 格式："provider/model_name"，LiteLLM 据此路由请求
        api_base=NVIDIA_API_BASE,         # NVIDIA API 的基础 URL
                                          # 默认："https://integrate.api.nvidia.com/v1"
        api_key=os.environ["NVIDIA_API_KEY"],  # 从环境变量读取 API Key
                                               # 在 Step 1 中已设置到 os.environ
    )

# ─────────────────────────────────────────────
# 4. 异步会话运行辅助函数
# ─────────────────────────────────────────────
# 这是整个 notebook 最核心的"胶水代码"：把用户输入 → ADK 结构化消息 → Runner 执行 → 打印输出。
# 设计为 async 函数，因为 ADK 的 session_service 和 runner 都是异步 API。
async def run_session(
    runner_instance: Runner,       # ADK Runner 实例：负责编排 Agent 执行
    user_queries: list[str] | str, # 用户输入：支持单条字符串或字符串列表
    session_id: str = "default",   # 会话标识：相同 session_id 会复用上下文历史
):
    """
    创建或复用指定 session，逐条发送用户消息，并打印模型的最终回复。
    
    执行流程：
    1. 尝试创建新 session（如果 session_id 已存在则捕获异常）
    2. 异常时回退到 get_session() 读取已有 session
    3. 将每条用户消息包装成 ADK Content 对象
    4. 通过 runner.run_async() 触发 Agent 执行
    5. 过滤事件流，只输出最终回复文本
    """
    print(f"\n### Session: {session_id}")
    
    # ── 4.1 Session 创建或复用 ──
    # ADK 的 session 是持久化上下文的载体，包含对话历史、状态(state)等。
    # create_session() 在 session_id 已存在时会抛异常，所以用 try/except 做"存在则复用"逻辑。
    try:
        session = await session_service.create_session(
            app_name=APP_NAME,      # 应用命名空间
            user_id=USER_ID,        # 用户隔离
            session_id=session_id,  # 会话唯一标识
        )
        print("   ↳ Created new session")
    except Exception:
        # 异常说明该 session_id 已存在，直接读取
        session = await session_service.get_session(
            app_name=APP_NAME,
            user_id=USER_ID,
            session_id=session_id,
        )
        print("   ↳ Reusing existing session")
    
    # ── 4.2 统一输入格式：单条字符串 → 列表 ──
    # 这样调用者既可以传 "hello" 也可以传 ["hi", "how are you"]，函数内部统一处理
    if isinstance(user_queries, str):
        user_queries = [user_queries]
    
    # ── 4.3 逐条处理用户消息 ──
    for query in user_queries:
        print(f"\nUser > {query}")
        
        # ADK 不接收裸字符串，必须包装成 types.Content 结构化对象
        # Content 包含 role（角色）和 parts（内容片段列表）
        query_content = types.Content(
            role="user",                      # 标识这是用户说的话
            parts=[types.Part(text=query)],   # text 类型的内容片段
        )
        
        # ── 4.4 调用 Runner 执行 Agent ──
        # run_async() 返回异步事件流（AsyncIterator[Event]）
        # Event 是 ADK 执行过程中的原子单元：包含模型输出、工具调用、状态变更等
        async for event in runner_instance.run_async(
            user_id=USER_ID,           # 用户标识
            session_id=session.id,     # 使用上面创建/获取的 session 对象的真实 ID
            new_message=query_content, # 用户新消息
        ):
            # ── 4.5 过滤并打印最终回复 ──
            # ADK 的事件流包含中间过程（思考、工具调用等），我们只关心最终结果：
            # is_final_response() = True 表示这是 Agent 对该轮对话的最终输出
            if event.is_final_response() and event.content and event.content.parts:
                text = event.content.parts[0].text
                
                # 防御性过滤：排除空内容和 "None" 字符串
                if text and text != "None":
                    # 打印格式：模型名 > 回复内容
                    print(f"{MODEL_NAME} > {text}")

# 模块加载完成提示
print("✅ Helper functions defined.")


✅ Helper functions defined.


## Section 1 · 初始化 MemoryService 与基础 Runner

In [9]:
# Step 7: 初始化 MemoryService
memory_service = InMemoryMemoryService()

print("✅ Memory service created:", memory_service.__class__.__name__)

✅ Memory service created: InMemoryMemoryService


In [10]:
# Step 8: 创建一个基础 agent（先不加 memory 工具）
user_agent = LlmAgent(
    model=make_nvidia_model(),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words.",
)

print("✅ Base agent created.")

✅ Base agent created.


In [11]:
# Step 9: 创建 SessionService 和 Runner，并把 MemoryService 接进去
session_service = InMemorySessionService()

runner = Runner(
    agent=user_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)

print("✅ Runner created with memory support.")

✅ Runner created with memory support.


## Section 2 · 手动把 Session 写入 Memory

In [12]:
# Step 10: 先进行一轮对话，填充 session 数据
await run_session(
    runner,
    "My favorite color is blue-green. Can you write a Haiku about it?",
    "conversation-01",
)


### Session: conversation-01
   ↳ Created new session

User > My favorite color is blue-green. Can you write a Haiku about it?
openai/gpt-oss-120b > The system says answer user questions in simple words. User asks: "My favorite color is blue-green. Can you write a Haiku about it?" Should respond with a haiku about blue-green, using simple words. Also follow simple words. Provide a haiku (3 lines, 5-7-5 syllables). Let's craft: "Blue-green waves whisper / Morning light dances soft / Calm sea sighs." Count syllables: "Blue-green waves whis-per" = Blue (1) green (1) waves (1) whis (1) per (1) =5? Actually "whisper" 2 syllables (whis-per). So count: Blue (1) green (1) waves (1) whis (1) per (1) =5. Good. Second line: "Morning light dances soft" -> Morning (2) light (1) dan-ces (2) soft (1) =6? Actually "Morning" 2, "light"1=3, "dances" 2=>5, "soft"1=>6. Need 7. Add "in" maybe: "Morning light dances in soft" -> Morning(2)+light(1)=3+dances(2)=5+in(1)=6+soft(1)=7 good. Third line: "Calm sea

In [13]:
# ============================================
# Step 11: 读取持久化 Session，查看其中的事件历史
# ============================================

# ── 1. 从 SessionService 中读取指定 Session ──
# get_session() 是异步方法，通过 (app_name, user_id, session_id) 三元组
# 唯一检索一个 session 对象。这里读取的是 Step 10 中创建的 "conversation-01"。
session = await session_service.get_session(
    app_name=APP_NAME,            # 应用名称，必须与创建时一致
    user_id=USER_ID,              # 用户标识，必须与创建时一致
    session_id="conversation-01", # 目标会话 ID：读取哪段对话的历史
)

# ── 2. 遍历 Session 中的事件列表 ──
# session.events 是一个按时间顺序排列的 Event 对象列表。
# 每个 Event 代表对话中的一个原子动作：用户发言、模型回复、工具调用、状态变更等。
print("📝 Session contains:")
for event in session.events:
    
    # ── 2.1 安全提取角色（role）信息 ──
    # 用 getattr 做防御式访问，避免 AttributeError。
    # content.role 标识这条消息是谁发出的："user"（用户）、"model"（模型）或 "tool"（工具）。
    role = getattr(getattr(event, "content", None), "role", None)
    
    # ── 2.2 安全提取文本内容 ──
    text = ""
    if event.content and event.content.parts:
        # event.content.parts 是内容片段列表，通常文本消息只有一个 text 类型的 part。
        # 这里取第一个 part（对于普通对话足够，富媒体消息可能有多个 parts）。
        first_part = event.content.parts[0]
        
        # 同样使用 getattr 防御式提取 text 属性。
        # 某些 part 类型（如 function_call）可能没有 text 字段，此时返回空字符串。
        text = getattr(first_part, "text", "") or ""
    
    # ── 2.3 格式化输出 ──
    # 只打印前 120 个字符，防止长回复刷屏，同时保留可读性。
    print(f"  {role}: {text[:120]}...")


📝 Session contains:
  user: My favorite color is blue-green. Can you write a Haiku about it?...
  model: The system says answer user questions in simple words. User asks: "My favorite color is blue-green. Can you write a Haik...


In [14]:
# Step 12: 把整个 session 手动写入 memory
await memory_service.add_session_to_memory(session)

print("✅ Session added to memory!")

✅ Session added to memory!


## Section 3 · 给 Agent 加上 Memory 检索工具

这里使用官方文档中的 `LoadMemoryTool()`：  
它会在 **Agent 自己认为需要回忆历史信息** 时再去搜索 memory。

In [15]:
# Step 13: 创建带 LoadMemoryTool 的 agent
user_agent = LlmAgent(
    model=make_nvidia_model(),
    name="MemoryDemoAgent",
    instruction=(
        "Answer user questions in simple words. "
        "If you need to recall past conversations, use the memory tool."
    ),
    tools=[LoadMemoryTool()],
)

print("✅ Agent with LoadMemoryTool created.")

✅ Agent with LoadMemoryTool created.


In [16]:
# Step 14: 用新 agent 重建 Runner，并测试记忆检索
runner = Runner(
    agent=user_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)

await run_session(
    runner,
    "What is my favorite color?",
    "color-test",
)



### Session: color-test
   ↳ Created new session

User > What is my favorite color?
openai/gpt-oss-120b > Your favorite color is blue‑green.


In [17]:
# Step 15: 再创建一个新的 session，写入“生日”信息
await run_session(
    runner,
    "My birthday is on March 15th.",
    "birthday-session-01",
)


### Session: birthday-session-01
   ↳ Created new session

User > My birthday is on March 15th.
openai/gpt-oss-120b > User says "My birthday is on March 15th." Probably want to store this info. The user didn't ask a question. We could respond acknowledging and perhaps store in memory? We have only load_memory, not save. We can't store. Perhaps just acknowledge. The instruction: answer user questions simply. No question. So just respond acknowledging.


In [18]:
# Step 16: 手动把生日 session 写入 memory
birthday_session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id="birthday-session-01",
)

await memory_service.add_session_to_memory(birthday_session)

print("✅ Birthday session saved to memory!")

✅ Birthday session saved to memory!


In [19]:
# Step 17: 在一个不同的 session 中回忆生日
await run_session(
    runner,
    "When is my birthday?",
    "birthday-session-02",
)


### Session: birthday-session-02
   ↳ Created new session

User > When is my birthday?
openai/gpt-oss-120b > Your birthday is on March 15th. 🎉


In [20]:
# Step 18: 直接在代码里手动搜索 memory
search_response = await memory_service.search_memory(
    app_name=APP_NAME,
    user_id=USER_ID,
    query="What is the user's favorite color?",
)

print("🔍 Search Results:")
print(f"  Found {len(search_response.memories)} relevant memories")
print()

for idx, memory in enumerate(search_response.memories, start=1):
    print(f"--- Memory #{idx} ---")
    author = getattr(memory, "author", None)
    print("Author:", author)

    if hasattr(memory, "content") and memory.content and memory.content.parts:
        first_part = memory.content.parts[0]
        text = getattr(first_part, "text", "") or ""
        print("Text:", text[:300])
    else:
        print("Memory object:", memory)
    print()

🔍 Search Results:
  Found 4 relevant memories

--- Memory #1 ---
Author: user
Text: My favorite color is blue-green. Can you write a Haiku about it?

--- Memory #2 ---
Author: MemoryDemoAgent
Text: The system says answer user questions in simple words. User asks: "My favorite color is blue-green. Can you write a Haiku about it?" Should respond with a haiku about blue-green, using simple words. Also follow simple words. Provide a haiku (3 lines, 5-7-5 syllables). Let's craft: "Blue-green waves 

--- Memory #3 ---
Author: user
Text: My birthday is on March 15th.

--- Memory #4 ---
Author: MemoryDemoAgent
Text: User says "My birthday is on March 15th." Probably want to store this info. The user didn't ask a question. We could respond acknowledging and perhaps store in memory? We have only load_memory, not save. We can't store. Perhaps just acknowledge. The instruction: answer user questions simply. No ques



## Section 4 · 用 Callback 自动写入 Memory

这里使用官方文档里的思路：

- `after_agent_callback` 在每次 agent 回答后触发
- callback 自动调用 `add_session_to_memory()`
- 再配合 `PreloadMemoryTool()`，每轮开头都会先加载 memory

In [21]:
# ============================================
# Step 19: 定义自动保存 session 到 memory 的 callback
# ============================================
# 本段代码创建一个"钩子函数"（callback），在 Agent 每轮执行结束后自动触发，
# 将当前完整的对话 session 写入 memory_service，实现"边聊边记"的自动持久化。

# ── Callback 函数定义 ──
# 在 ADK 中，callback 是一种事件钩子机制。你可以在 Agent 生命周期的特定节点
# （如：before_model、after_model、before_tool、after_tool 等）挂载自定义逻辑。
# 这里定义的函数会在"每轮 Agent 执行完成后"被 ADK 框架自动调用。
async def auto_save_to_memory(callback_context):
    """
    自动将当前 session 保存到记忆服务中。
    
    参数:
        callback_context: ADK 传入的回调上下文对象，包含当前执行的所有上下文信息
                         （如 session、memory_service、agent 实例、用户消息等）
    """
    
    # ── 访问 InvocationContext ──
    # callback_context 是一个轻量级的包装器，真正的执行上下文藏在 _invocation_context 中。
    # _invocation_context 是当前这次"调用/推理"的完整上下文，包含：
    #   - session: 当前对话 session（含所有历史事件）
    #   - memory_service: 记忆服务实例（负责长期记忆的存储与检索）
    #   - user_id, session_id: 用户和会话标识
    invocation_context = callback_context._invocation_context
    
    # ── 调用记忆服务的 add_session_to_memory ──
    # 这行代码是整个自动记忆功能的核心：
    #   1. 从 invocation_context 中取出当前 session 对象
    #   2. 调用 memory_service.add_session_to_memory(session) 将其序列化并存储
    # 
    # 存储后，这段对话历史就变成了"长期记忆"，后续新 session 可以通过记忆检索工具
    # （如 LoadMemoryTool / PreloadMemoryTool）查询到这些内容。
    await invocation_context.memory_service.add_session_to_memory(
        invocation_context.session
    )

print("✅ Callback created.")


✅ Callback created.


In [ ]:
# ============================================
# Step 20: 创建自动记忆 Agent
# ============================================

# ── 创建 LlmAgent 实例 ──
auto_memory_agent = LlmAgent(
    # ── 1. 模型配置 ──
    # 调用 Step 6 定义的工厂函数，返回配置好的 LiteLlm 实例（连接 NVIDIA API）。
    # 每次 Agent 需要调用 LLM 时，都会使用这个模型。
    model=make_nvidia_model(),
    
    # ── 2. Agent 名称 ──
    # 在日志、调试信息、session 事件中标识这个 Agent 的身份。
    name="AutoMemoryAgent",
    
    # ── 3. 系统指令（System Instruction）──
    # 这是发给 LLM 的"身份设定"，决定了 Agent 的行为风格和可用工具的使用策略。
    # 这里明确告诉模型：回答要清晰，并且在遇到关于先前对话的问题时，使用已预加载的记忆。
    instruction=(
        "Answer user questions clearly. "
        "Use preloaded memory when it helps answer questions about prior conversations."
    ),
    
    # ── 4. 工具列表 ──
    # PreloadMemoryTool 是 ADK 提供的记忆检索工具。
    # 当 Agent 判断需要查询历史对话时，会自动调用它从 memory_service 中检索相关记忆片段。
    # 检索结果会被注入到当前上下文中，供模型参考。
    tools=[PreloadMemoryTool()],
    
    # ── 5. Agent 执行后的回调钩子 ──
    # after_agent_callback 在 Agent 的"一整轮执行"完全结束后触发。
    # 
    # ⚠️ 注意区分：
    #   - after_model_callback: 只在 LLM 生成一次回复后触发（不含工具调用）
    #   - after_agent_callback: 在整个 Agent 轮次结束后触发（可能包含多次模型调用 + 工具调用）
    # 
    # 这里用 after_agent_callback 能确保：即使 Agent 中间调用了 PreloadMemoryTool（工具执行），
    # 最终也会把"完整的用户提问 + 记忆检索结果 + 模型回复"一并保存到长期记忆。
    after_agent_callback=auto_save_to_memory,
)

print("✅ Auto-memory agent created.")


✅ Auto-memory agent created.


In [ ]:
# ============================================
# Step 21: 创建新的 Runner（自动记忆版）
# ============================================

# ── 创建 Runner 实例 ──
# Runner 是 ADK 的"执行引擎"和"指挥中心"。
# 它负责：接收用户输入 → 找到对应 Agent → 管理 Session 生命周期 → 分发事件流。
auto_runner = Runner(
    # ── 1. Agent 实例 ──
    # 指定由哪个 Agent 处理用户请求。这里传入的是 Step 20 创建的 auto_memory_agent，
    # 它自带 PreloadMemoryTool 和 after_agent_callback 自动保存逻辑。
    agent=auto_memory_agent,
    
    # ── 2. 应用名称 ──
    # 命名空间标识。同一个 app_name 下的所有 session 和记忆共享同一个逻辑应用。
    # 必须与 Step 6 定义的 APP_NAME 一致，否则 session 和记忆会"找不到彼此"。
    app_name=APP_NAME,
    
    # ── 3. Session 服务 ──
    # 负责 session 的创建、读取、更新。可以是：
    #   - InMemorySessionService(): 内存型，进程结束数据丢失（演示/测试用）
    #   - DatabaseSessionService(): 数据库型，持久化存储（生产用）
    # 
    # Runner 通过它存取对话历史，确保多轮对话上下文连贯。
    session_service=session_service,
    
    # ── 4. 记忆服务（⭐ 自动记忆的核心）──
    # memory_service 是长期记忆的存储后端，通常基于向量数据库（如 Vertex AI Vector Search、
    # Chroma、或 ADK 内置的简单实现），负责：
    #   1. 存储：add_session_to_memory() 把对话转成向量/文本存入记忆库
    #   2. 检索：PreloadMemoryTool 通过语义相似度搜索相关历史片段
    # 
    # ⚠️ 如果不传 memory_service：
    #   - auto_save_to_memory 回调里的 memory_service 会为 None → 保存失败
    #   - PreloadMemoryTool 无法检索到任何历史 → 记忆功能形同虚设
    memory_service=memory_service,
)

print("✅ Auto-memory runner created.")


✅ Auto-memory runner created.


In [ ]:
# Step 22: 测试自动保存 + 跨 session 记忆检索
await run_session(
    auto_runner,
    "I gifted a new toy to my nephew on his 1st birthday!",
    "auto-save-test",
)

await run_session(
    auto_runner,
    "What did I gift my nephew?",
    "auto-save-test-2",
)


### Session: auto-save-test
   ↳ Created new session

User > I gifted a new toy to my nephew on his 1st birthday!
openai/gpt-oss-120b > We need to respond. The user statement: "I gifted a new toy to my nephew on his 1st birthday!" Likely they share info, maybe want comment. As per instruction: answer user questions clearly, use memory when helpful. No question asked. We can acknowledge and perhaps ask a follow-up or give a comment. Should keep simple words. So respond with a friendly acknowledgement, maybe ask how the nephew liked it. Use simple words.

### Session: auto-save-test-2
   ↳ Created new session

User > What did I gift my nephew?
openai/gpt-oss-120b > We need to answer using memory. The past conversation says: "I gifted a new toy to my nephew on his 1st birthday!" So the gift is a new toy. They didn't specify which toy. So answer: You gave a new toy. Possibly we can restate. Use simple words.


## 完成

这个 notebook 现在已经完整覆盖了：

- 初始化 `MemoryService`
- 手动 `add_session_to_memory()`
- `LoadMemoryTool()` 让 agent 按需检索
- `search_memory()` 手动搜索
- `after_agent_callback + PreloadMemoryTool()` 自动保存与自动预加载

如果你还要继续，我可以把你上传的 **day-2a / day-2b** 也按同样方式改成 **NVIDIA API + Colab + 完整代码单元** 的 `.ipynb` 文件。